# Multi-dataset validation (industry plan, phase 1)

This notebook is the exact procedure behind `docs/RESULTS_PUBLIC.md`. It runs the **same** MarginGraph pipeline that was built on Online Retail II on four other public retail datasets and asks one question: **does the LightGBM reranker beat the best simple method on stores it was not designed on?**

**What runs, per dataset** (all repo commands, nothing special-cased):

1. `scripts/public_datasets.py convert`: raw files → `orders.csv` + `products.csv` (connector schema), keeping the 5,000 most-ordered products.
2. `margingraph sync csv`: cleans the files and writes a store `config.yaml` with 70% / 85% time cuts (train / validation / test).
3. `scripts/public_datasets.py simulate-erp`: these datasets have no unit cost or stock, so Stage 2 uses the **simulated** cost and stock, exactly like UCI.
4. `margingraph run`: experiments E1 to E11.
   * **Stage 1:** candidate pool of 150 from popularity, co-purchase, item-kNN, EASE, ALS, time-decayed EASE, the customer's own history, TF-IDF text and fresh trends; then a **LightGBM LambdaRank** reranker trained on validation baskets.
   * **Stage 2:** **PuLP MILP** picks at most 2 add-ons with a 15% margin floor (profit numbers here are simulated).
5. `scripts/public_datasets.py compare`: LightGBM vs the best baseline (picked on validation, scored on test) with a paired bootstrap 95% CI.

**Gate (fixed before running, from `docs/industry-plan/01-multi-dataset-validation.md`):** "generalizes" means LightGBM's Hit@10 beats the best baseline with the 95% CI above zero on most datasets that have at least 1,000 test baskets.

**Where to run:** a Kaggle CPU notebook (about 30 GB RAM, add the datasets as inputs) or any machine with about 8 GB free. The cloud run that produced the committed results used 4 CPUs and 15 GB; per-dataset time and peak RAM are printed below and saved to `run_stats.json`.

In [ ]:
import os, sys, json, subprocess, time, resource, hashlib, zipfile, shutil, urllib.request
from pathlib import Path

# Run from the repo root. On Kaggle / Colab, clone it first.
if not Path("scripts/public_datasets.py").exists():
    if not Path("MarginGraph").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Sanskar1724/MarginGraph.git"], check=True)
    os.chdir("MarginGraph")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
ROOT = Path.cwd()
print("repo:", ROOT)

In [ ]:
# ---- settings -----------------------------------------------------------------
RAW = Path(os.environ.get("MG_PUBLIC_RAW", "data/public"))   # raw downloads, one folder per dataset (git-ignored)
TOP_ITEMS = 5000                    # keep the N most-ordered products (EASE memory grows with N^2)
CONVERT_OPTS = {                    # per-dataset converter options from the phase-1 plan
    "olist": [],
    "tafeng": [],
    "dunnhumby": [],
    "instacart": ["--max-customers", "20000"],   # 206k users is too big for 8 GB; sample 20k
}
REUSE = False                       # True = skip datasets that already have results/public/<name>/summary.json
KAGGLE_INPUT = Path("/kaggle/input")

## 1. Get the data

* **Olist**: downloaded from Olist's own GitHub repository (`olist/work-at-olist-data`), the same files as the Kaggle dataset `olistbr/brazilian-ecommerce` (99,441 orders, 112,650 item rows).
* **Ta Feng**: the Kaggle file `ta_feng_all_months_merged.csv` (`chiranjivdas09/ta-feng-grocery-dataset`, 817,741 rows), here fetched from a GitHub mirror of that Kaggle upload (`RecoHut-Datasets/tafeng`, branch `v1`) and checked by SHA-256.
* **Dunnhumby** and **Instacart**: Kaggle only. On Kaggle, add them as notebook inputs. Elsewhere, download them by hand and unzip into `data/public/dunnhumby/` and `data/public/instacart/`:
  * https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey (needs `transaction_data.csv`, `product.csv`)
  * https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis (needs `orders.csv`, `order_products__prior.csv`, `order_products__train.csv`, `products.csv`, `departments.csv`)

A dataset whose files are missing is skipped, not faked.

In [ ]:
def fetch(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# Olist: official GitHub copy
OLIST = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"
for f in ["olist_orders_dataset", "olist_order_items_dataset", "olist_products_dataset",
          "olist_customers_dataset", "product_category_name_translation"]:
    fetch(OLIST + f + ".csv", RAW / "olist" / f"{f}.csv")

# Ta Feng: mirror of the Kaggle upload, verified by hash
TAFENG_SHA = "1d575e5d0b7207d7706d22ca56c7535886fff8175ca5537a310333a4ab7a7b67"
tf = RAW / "tafeng" / "ta_feng_all_months_merged.csv"
if not tf.exists():
    tmp = RAW / "tafeng" / "_mirror"
    if not tmp.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "v1",
                        "https://github.com/RecoHut-Datasets/tafeng", str(tmp)], check=True)
    zipfile.ZipFile(tmp / "ta-feng-grocery-dataset.zip").extractall(RAW / "tafeng")
assert sha256(tf) == TAFENG_SHA, "Ta Feng file differs from the verified copy"

# Kaggle inputs, when running on Kaggle
for name, slug in [("dunnhumby", "dunnhumby-the-complete-journey"), ("instacart", "instacart-market-basket-analysis")]:
    k = KAGGLE_INPUT / slug
    if k.exists() and not (RAW / name).exists():
        (RAW / name).symlink_to(k)

NEEDS = {
    "olist": ["olist_orders_dataset.csv", "olist_order_items_dataset.csv", "olist_customers_dataset.csv", "olist_products_dataset.csv"],
    "tafeng": ["ta_feng_all_months_merged.csv"],
    "dunnhumby": ["transaction_data.csv", "product.csv"],
    "instacart": ["orders.csv", "order_products__prior.csv", "order_products__train.csv", "products.csv"],
}
def have(name):
    found = {p.name.lower() for p in (RAW / name).rglob("*")} if (RAW / name).exists() else set()
    return all(f in found for f in NEEDS[name])

AVAILABLE = [n for n in CONVERT_OPTS if have(n)]
print("datasets with files:", AVAILABLE)
print("missing (skipped):", [n for n in CONVERT_OPTS if n not in AVAILABLE])

## 2. Convert → sync → simulate-erp → run

`sh()` prints each command before running it, so the output below is the full log. `run_stats.json` records the run time and the peak memory of the `margingraph run` process.

In [ ]:
def sh(*cmd):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-25:]))          # last lines of each step's log
    if r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {' '.join(cmd)}")
    return time.time() - t

def run_dataset(name, extra=(), store=None, results=None, simulate=True):
    store = Path(store or f"data/store/{name}")
    results = Path(results or f"results/public/{name}")
    if REUSE and (results / "summary.json").exists():
        print(f"{name}: reusing {results}"); return
    conv = RAW / name / ("mg" if simulate else "mg_" + store.name)
    sh(sys.executable, "scripts/public_datasets.py", "convert", name, "--src", RAW / name, "--out", conv,
       "--top-items", TOP_ITEMS, *extra)
    sh("margingraph", "sync", "csv", "--orders", conv / "orders.csv", "--products", conv / "products.csv", "--out", store)
    if simulate:
        sh(sys.executable, "scripts/public_datasets.py", "simulate-erp", store / "config.yaml")
    before = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss
    secs = sh("margingraph", "--config", store / "config.yaml", "run", "--results", results,
              "--artifacts", f"data/artifacts/{store.name}")
    peak = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss   # KB on Linux; max over all children so far
    shutil.copy(conv / "convert_stats.json", results / "convert_stats.json")
    stats = {"seconds": round(secs), "peak_rss_mb_upper_bound": peak // 1024, "cpus": os.cpu_count(),
             "convert_options": ["--top-items", str(TOP_ITEMS), *map(str, extra)], "simulated_erp": simulate}
    (results / "run_stats.json").write_text(json.dumps(stats, indent=2))
    print(name, stats)

for name in AVAILABLE:
    print(f"\n===== {name} =====")
    run_dataset(name, CONVERT_OPTS[name])

### Ta Feng, second run: `ASSET` read as cost (Stage 2 only, unverified)

Ta Feng's `ASSET` column is often read as the line's cost, but the dataset does not document it. This run skips `simulate-erp` so Stage 2 uses `ASSET / AMOUNT` as unit cost (stock is still simulated). Its Stage 1 numbers are the same model on the same data; only the Stage 2 profit table differs. Label it **"ASSET read as cost, unverified"**.

In [ ]:
if "tafeng" in AVAILABLE:
    run_dataset("tafeng", ["--asset-as-cost"], store="data/store/tafeng_asset",
                results="results/public/tafeng_asset_cost", simulate=False)

## 3. Results

### LightGBM vs the best baseline (test set, Hit@10, paired bootstrap 95% CI)

In [ ]:
from IPython.display import Markdown, display
import pandas as pd

folders = ["results"] + [f"results/public/{n}" for n in AVAILABLE]
folders = [f for f in folders if Path(f, "summary.json").exists()]
table = subprocess.run([sys.executable, "scripts/public_datasets.py", "compare", *folders],
                       capture_output=True, text=True, check=True).stdout
display(Markdown(table))

### Hit@5 / Hit@10 / NDCG@10 differences, and what each dataset looks like

In [ ]:
rows = []
for f in folders:
    d = Path(f)
    s = json.loads((d / "summary.json").read_text())
    pr = pd.read_csv(d / "e5_paired.csv")
    a = json.loads((d / "e1_audit.json").read_text())
    row = {"results": f, "best baseline": s.get("best_baseline_on_val"),
           "pool recall": round(100 * s["e4_test_pool_recall"], 1),
           "median basket": a["baskets"]["median_basket_items"],
           "share baskets 2+": round(100 * a["baskets"]["share_baskets_2plus"], 1),
           "co-purchase lift vs shuffled": round(a["copurchase_vs_shuffled_on_val"]["lift_hit@10_vs_shuffled"], 1)}
    for _, r in pr.iterrows():
        row[f"{r.metric} diff [CI]"] = f"{100*r['diff']:+.1f} [{100*r.ci_low:+.1f}, {100*r.ci_high:+.1f}]"
    rows.append(row)
pd.DataFrame(rows).set_index("results")

### Top LightGBM features per dataset

In [ ]:
for f in folders:
    imp = pd.read_csv(Path(f) / "e5_importance.csv")
    col = "gain" if "gain" in imp.columns else imp.columns[-1]
    print(f, "->", ", ".join(imp.sort_values(col, ascending=False).iloc[:6, 0].astype(str)))

### Stage 2 (profit is SIMULATED: simulated cost and stock, or Ta Feng's unverified ASSET cost)

In [ ]:
extra = ["results/public/tafeng_asset_cost"] if Path("results/public/tafeng_asset_cost/summary.json").exists() else []
for f in folders[1:] + extra:
    p = Path(f) / "e8_paired_vs_ranker_top2.csv"
    if p.exists():
        print("\n", f)
        display(pd.read_csv(p).round(4))

In [ ]:
for f in folders[1:] + extra:
    p = Path(f, "run_stats.json")
    if p.exists():
        print(f, json.loads(p.read_text()))